# 02 – Data integration

**Goal:** Combine the four sources into consistent tables with common keys and document every join: key, rows before and after, and the reason for every lost record.
**Input:** `data/raw/`, `data/interim/jolpica_*.parquet`, `data/interim/wikipedia_circuits.parquet` · **Output:** `data/interim/{circuits,races,drivers,driver_map,results,laps,weather}.parquet`, `reports/logs/integration_log.csv`
**Assessment items:** P2 (preparation and integration of at least two sources, documented)

| Step | Left | Right | Key | Method |
|---|---|---|---|---|
| 1 | FastF1 sessions | Jolpica calendar | season + round | exact join, then name/date check |
| 2 | Jolpica circuits | GeoJSON locations | coordinates | nearest point (haversine) ≤ 5 km |
| 2b | Jolpica circuits | Wikipedia circuit list | circuit name (+ town) | fuzzy match (rapidfuzz) + manual correction table |
| 3 | FastF1 drivers | Jolpica drivers | season + round + car number | exact join, check against FastF1 DriverId and abbreviation |
| 4 | Jolpica results | Jolpica qualifying | season + round + driverId | left join |
| 5 | FastF1 laps | driver mapping (step 3) | season + round + car number | exact join |
| 6 | FastF1 weather | laps | session time | as-of join (backward) when building pairs, notebook 04 |

In [1]:
import sys
from pathlib import Path

# Make the repository root importable, whether Jupyter starts in the root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [2]:
from src.config import INTERIM_DIR, PILOT_SEASONS
from src.integrate import integrate

SEASONS = PILOT_SEASONS
jolpica = {name: pd.read_parquet(INTERIM_DIR / f"jolpica_{name}.parquet")
           for name in ("races", "results", "qualifying")}
wiki = pd.read_parquet(INTERIM_DIR / "wikipedia_circuits.parquet")

tables, log = integrate(SEASONS, jolpica, wiki)

## 1. Integration log

In [3]:
with pd.option_context("display.max_colwidth", None):
    display(log.style.hide(axis="index").set_properties(**{"text-align": "left"}))

step,rows_before,rows_after,change,note
1 races: FastF1 sessions <-> Jolpica calendar,44,44,0,"key (season, round)"
1b races: consistency check name/date,44,44,0,0 races with name similarity < 80 or date difference > 1 day
2 circuits: Jolpica circuitId <-> GeoJSON location (nearest point),26,26,0,matched if distance <= 5.0 km; unmatched: []; max matched distance 1.03 km
2b circuits: Jolpica circuit name <-> Wikipedia list (fuzzy),26,26,0,"rapidfuzz token_set_ratio >= 85, or >= 50 with identical town, else manual table; {'fuzzy name': 24, 'fuzzy name + town': 2}"
3 drivers: FastF1 car number per race <-> Jolpica driverId,880,880,0,"key (season, round, number); unmatched 0; driverId differs from FastF1 DriverId: 0; abbreviation differs from Jolpica code: 0"
"3b driver_map: unique (season, abbreviation)",43,43,0,no conflicts
4 results: race result + qualifying position,880,880,0,"left join on (season, round, driverId); 1 drivers without qualifying position (quali_gap will be missing for them)"
"5 laps: FastF1 laps -> driver_id, seconds",47333,47333,0,"driver not mapped: 0; lap number missing: 0; duplicate (race, driver, lap): 0; info: position missing 66, IsAccurate=False 6888"
6 weather: FastF1 weather samples,7342,7342,0,"rows without time or duplicate (race, time) removed; joined to pairs later with merge_asof (backward) on lap start time"


## 2. Circuit matching (GeoJSON and Wikipedia)

In [4]:
tables["circuit_matching"][["circuit_id", "circuit_name", "locality", "geojson_id",
                            "geo_distance_km", "wiki_name", "wiki_match", "wiki_score",
                            "circuit_type", "length_km", "turns"]]

,circuit_id,circuit_name,locality,geojson_id,geo_distance_km,wiki_name,wiki_match,wiki_score,circuit_type,length_km,turns
0,bahrain,Bahrain International Circuit,Sakhir,bh-2002,0.22,Bahrain International Circuit,fuzzy name,100.000000,race,5.412,15
1,imola,Autodromo Enzo e Dino Ferrari,Imola,it-1953,0.44,Autodromo Internazionale Enzo e Dino Ferrari,fuzzy name,100.000000,race,4.909,19
2,portimao,Autódromo Internacional do Algarve,Portimão,pt-2008,0.57,Algarve International Circuit,fuzzy name + town,73.015873,race,4.653,15
3,catalunya,Circuit de Barcelona-Catalunya,Barcelona,es-1991,0.21,Circuit de Barcelona-Catalunya,fuzzy name,100.000000,race,4.657,14
4,monaco,Circuit de Monaco,Monte Carlo,mc-1929,0.72,Circuit de Monaco,fuzzy name,100.000000,street,3.337,19
5,baku,Baku City Circuit,Baku,az-2016,1.03,Baku City Circuit,fuzzy name,100.000000,street,6.003,20
6,ricard,Circuit Paul Ricard,Le Castellet,fr-1969,0.27,Circuit Paul Ricard,fuzzy name,100.000000,race,5.842,15
7,red_bull_ring,Red Bull Ring,Spielberg,at-1969,0.46,Red Bull Ring,fuzzy name,100.000000,race,4.318,10
8,silverstone,Silverstone Circuit,Silverstone,gb-1948,0.73,Silverstone Circuit,fuzzy name,100.000000,race,5.891,18
9,hungaroring,Hungaroring,Budapest,hu-1986,0.47,Hungaroring,fuzzy name,100.000000,race,4.381,14


## 3. Data quality of the lap table

In [5]:
laps = tables["laps"]
quality = pd.Series({
    "laps": len(laps),
    "position missing": int(laps["position"].isna().sum()),
    "lap time missing": int(laps["lap_time_s"].isna().sum()),
    "IsAccurate = False": int((~laps["is_accurate"].astype(bool)).sum()),
    "pit-in laps": int(laps["pit_in"].sum()),
    "pit-out laps": int(laps["pit_out"].sum()),
})
quality.to_frame("count")

,count
laps,47333
position missing,66
lap time missing,245
IsAccurate = False,6888
pit-in laps,1682
pit-out laps,1659


In [6]:
laps["compound"].value_counts(dropna=False).to_frame("laps")

,laps
compound,
HARD,18516
MEDIUM,17735
SOFT,7104
INTERMEDIATE,3176
WET,408
None,334
UNKNOWN,60


## 4. Resulting tables

In [7]:
pd.Series({name: len(df) for name, df in tables.items()
           if name != "circuit_matching"}).to_frame("rows")

,rows
circuits,26
races,44
drivers,26
driver_map,43
results,880
laps,47333
weather,7342
